# Training YOLO26n — dataset v3 canonico

Questo notebook addestra da `models/yolo26n.pt` sullo split pubblico Train, usa Validation per selezionare il checkpoint e valuta TEST-ID e TEST-OOD soltanto dopo il training. Il checkpoint v2 non viene riutilizzato perché potrebbe aver visto immagini ora assegnate a TEST-ID.

Le 240 immagini operative restano esclusivamente in TEST-OOD. Il modello conserva tutte le 80 classi COCO richieste dal runtime, anche se il dataset annota soltanto le sei classi operative.

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import sys

import torch
import yaml
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "phenocam").is_dir():
    raise RuntimeError("Avvia il notebook dalla radice del repository o da notebooks/")
sys.path.insert(0, str(ROOT))

from dataset.builder.partition import verify_artifact

DATASET = ROOT / "dataset" / "dataset-v3"
BASE_MODEL = ROOT / "models" / "yolo26n.pt"
WORK = ROOT / "output" / "training-v3"
RUNS = WORK / "runs"
V3_PT = ROOT / "models" / "yolo26n-v3.pt"
V3_ONNX = ROOT / "models" / "yolo26n-v3.onnx"
EPOCHS, IMAGE_SIZE, BATCH, SEED = 30, 640, 8, 42
DEVICE = "mps" if torch.backends.mps.is_available() else 0 if torch.cuda.is_available() else "cpu"

if not BASE_MODEL.is_file():
    raise RuntimeError("Checkpoint COCO models/yolo26n.pt assente")
WORK.mkdir(parents=True, exist_ok=True)
print({"torch": torch.__version__, "device": DEVICE, "epochs": EPOCHS, "batch": BATCH})

## Verifica e configurazione

Il verificatore controlla checksum, conteggi, associazioni immagine/label, identità, gruppi, camere e leakage pHash. Il file YAML di lavoro espande i nomi a tutte le 80 classi COCO senza modificare le label del dataset.

In [ ]:
verification = verify_artifact(DATASET)
if verification["status"] != "passed" or verification["split_images"] != {"train": 1600, "val": 200, "test_id": 200, "test_ood": 240}:
    raise RuntimeError("Il dataset v3 non rispetta il contratto canonico")

checkpoint = YOLO(BASE_MODEL)
coco_names = dict(checkpoint.names)
target_names = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
if len(coco_names) != 80 or any(coco_names[class_id] != name for class_id, name in target_names.items()):
    raise RuntimeError("Il checkpoint non espone l'inventario COCO atteso")

data_yaml = WORK / "dataset-v3-80-classes.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": ["images/test/id", "images/test/ood"],
    "names": coco_names,
}, sort_keys=False), encoding="utf-8")
print(json.dumps(verification, indent=2, sort_keys=True))

## Training

`workers=0` evita processi figli fragili nei kernel macOS. Il miglior checkpoint su Validation viene copiato in `models/yolo26n-v3.pt`.

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(data_yaml), epochs=EPOCHS, patience=8, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, cache=False, seed=SEED, deterministic=True,
    project=str(RUNS), name="yolo26n-v3", exist_ok=True,
)
best_checkpoint = Path(model.trainer.best).resolve()
if not best_checkpoint.is_file():
    raise RuntimeError("Il training non ha prodotto best.pt")
shutil.copy2(best_checkpoint, V3_PT)
trained = YOLO(V3_PT)
if dict(trained.names) != coco_names:
    raise RuntimeError("Il modello v3 non conserva le 80 classi COCO")
print(f"Checkpoint v3: {V3_PT} ({V3_PT.stat().st_size / 1024 / 1024:.1f} MiB)")

## Metriche finali

Validation guida lo sviluppo. TEST-ID e TEST-OOD sono misurati separatamente e non devono essere usati per cambiare iperparametri, soglie o checkpoint.

In [ ]:
def summarize(metrics):
    return {
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "per_class_mAP50-95": {target_names[class_id]: float(metrics.box.maps[class_id]) for class_id in target_names},
    }

evaluation = {}
for split_name, split_path in (("validation", "images/val"), ("test-id", "images/test/id"), ("test-ood", "images/test/ood")):
    split_yaml = WORK / f"{split_name}.yaml"
    split_yaml.write_text(yaml.safe_dump({"path": str(DATASET), "val": split_path, "names": coco_names}, sort_keys=False), encoding="utf-8")
    metrics = trained.val(data=str(split_yaml), split="val", imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE, workers=0, plots=True, project=str(RUNS), name=split_name, exist_ok=True)
    evaluation[split_name] = summarize(metrics)
(WORK / "metrics-v3.json").write_text(json.dumps(evaluation, indent=2) + "\n", encoding="utf-8")
print(json.dumps(evaluation, indent=2))

## Export ONNX e contratto runtime

L'export è accettato soltanto se mantiene forma statica 640×640, output end-to-end e inventario COCO completo. Viene poi eseguita un'inferenza applicativa su un frame TEST-OOD, senza modificare il dataset.

In [ ]:
exported = Path(trained.export(format="onnx", opset=20, imgsz=IMAGE_SIZE, batch=1, dynamic=False)).resolve()
if exported != V3_ONNX.resolve():
    shutil.copy2(exported, V3_ONNX)

from phenocam.classes.selection import enabled_class_names, model_class_ids
from phenocam.inference.pipeline import process_image
from phenocam.inference.runtime import create_session, model_contract

session = create_session(V3_ONNX)
_, _, width, height, onnx_names = model_contract(session)
model_class_ids(onnx_names, enabled_class_names())
if onnx_names != coco_names or (width, height) != (IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError("Contratto ONNX v3 inatteso")
sample_input = next((DATASET / "images/test/ood").glob("*.jpg"))
sample_output = WORK / "runtime-sample.jpg"
elapsed = process_image(V3_ONNX, sample_input, sample_output, None)
print({"onnx": str(V3_ONNX), "sample": str(sample_output), "seconds": round(elapsed, 3)})

## Curve di training

In [ ]:
import matplotlib.pyplot as plt

history_path = RUNS / "yolo26n-v3" / "results.csv"
with history_path.open(newline="", encoding="utf-8") as stream:
    history = list(csv.DictReader(stream))
epochs = [int(row["epoch"]) for row in history]
map_all = [float(row["metrics/mAP50-95(B)"]) for row in history]
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [float(row["metrics/mAP50(B)"]) for row in history], label="mAP50")
axes[0].plot(epochs, map_all, label="mAP50-95")
axes[0].set(title="Qualità su Validation", xlabel="Epoch", ylabel="mAP", ylim=(0, 1)); axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].plot(epochs, [float(row["train/box_loss"]) for row in history], label="box train")
axes[1].plot(epochs, [float(row["val/box_loss"]) for row in history], label="box validation", linestyle="--")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Loss"); axes[1].legend(); axes[1].grid(alpha=0.2)
figure.tight_layout()
figure.savefig(WORK / "training-curves.png", dpi=150, bbox_inches="tight")
plt.show()